In [1]:
from file_management.x_foster_audio_files import deduplicate_bird_audio_data_enhanced

birds = ['or76bk11',
'bk34bk51',
'pk24bu3',
'bk94bk87',
'rd92gr54',
'ye25',
'gr99or87',
'pk15bk43',
'pu61bk41',
'rd95wh29',
'pu10bk94',
'pk54gr20',
'ye1',
'wh58pk1',
'or8bk49',
'gr26gr95',
'ye53ye83',
'ye27ye33',
'wh99pk7',
'wh94or25',
'wh94or11',
'wh89pk4',
'wh89or12',
'wh88pk84',
'wh88br85',
'wh83pk50',
'wh82pk41',
'wh80or5',
'wh75pk99',
'wh70pk50',
'wh69pk82',
'wh67pk20',
'wh66pk94',
'wh61pk3',
'wh59pk66',
'wh57pk61',
'wh54pk85',
'wh51or31',
'wh50pk34',
'wh3pk86',
'wh39or13',
'wh35pk25',
'wh28pk28',
'wh28or1',
'wh26pk33',
'wh23pk78',
'wh21pk15',
'wh11pk95',
'rd94wh47',
'rd93wh89',
'rd93wh64',
'rd88wh69',
'rd7wh75',
'rd78wh86',
'rd78wh62',
'rd77gr82',
'rd70wh90',
'rd63wh86',
'rd62wh35',
'rd61wh68',
'rd58wh32',
'rd4wh53',
'rd3wh90',
'rd34wh20',
'rd2wh49',
'rd28wh69',
'rd26wh66',
'rd26wh34',
'rd22wh20',
'rd10wh46',
'pu88bk78',
'pu81bk1',
'pu71wh42',
'pu64wh91',
'pu62wh53',
'pu51wh43',
'pu42bk10',
'pu39wh79',
'pu31bk90',
'pu27wh16',
'pu26wh18',
'pu25wh17',
'pu22wh17',
'pu20wh66',
'pu1wh51',
'pu1wh14',
'pu19wh65',
'pu16wh31',
'pu11bk28',
'pk97rd22',
'pk74gr77',
'pk43gr64',
'pk36gr15',
'pk1gr62',
'pk18gr79',
'or67or64',
'or41or32',
'or33or33',
'or28bu17',
'or14or56',
'or100or7',
'gr94gr1',
'gr89gr73',
'gr85gr34',
'gr82gr37',
'gr78gr76',
'gr70gr66',
'gr66gr43',
'gr65gr9',
'gr63gr76',
'gr57gr47',
'gr43gr62',
'gr38gr59',
'gr37gr41',
'gr34gr22',
'gr27gr28',
'gr25gr27',
'gr24gr26',
'bu95or85',
'bu92or63',
'bu91or64',
'bu82or62',
'bu81or63',
'bu71or60',
'bu69or87',
'bu66or8',
'bu65or7',
'bu50or44',
'bu49or82',
'bu48or83',
'bu28or67',
'bu21or96',
'bu17or20',
'bu11or68',
'bk9bk8',
'bk92bk23',
'bk89bk88',
'bk86bk62',
'bk84bk85',
'bk83bk73',
'bk83bk48',
'bk79bk82',
'bk73bk71',
'bk67bk44',
'bk65bk85',
'bk64bk65',
'bk61bk26',
'bk5bk60',
'bk5bk25',
'bk51bk59',
'bk50bk49',
'bk4bk95',
'bk4bk47',
'bk47bk40',
'bk43bk70',
'bk41bk14',
'bk38bk39',
'bk36bk37',
'bk34bk33',
'bk31bk89',
'bk30bk40',
'bk2bk10',
'bk29bk8',
'bk28bk68',
'bk20bk45',
'bk1bk9',
'bk1bk3',
'bk13bk12',
'bk100bk99',]

In [2]:
import logging
import sqlite3
from tools.dbquery import *
from x_foster_metadata import generate_bird_name_variations#, get_file_locations_for_birds

def get_file_locations_for_birds(cross_fostered_birds, txt_file_path="MacawAllDirsByBird.txt",
                                 db_path='2026-01-15-db.sqlite3'):
    """
    Parse the text file and return a dictionary mapping bird names to their file locations
    Only include male (M) or unknown (U) sex birds
    Now handles multiple variations of bird name formats for matching
    """
    bird_files = {}

    con = sqlite3.connect(db_path)
    cur = con.cursor()

    # First, get the bird names from cross_fostered_birds and filter by sex
    target_birds = {}
    target_bird_variations = {}  # Map variations back to original names

    for bird_name in cross_fostered_birds:
        try:
            uuid, exists = getUUIDfromBands(cur, bird_name)
            if exists:
                sex_query = "SELECT sex FROM birds_animal WHERE uuid=?"
                sex_result = cur.execute(sex_query, (uuid,))
                sex_row = sex_result.fetchone()
                sex = sex_row[0] if sex_row and sex_row[0] else 'U'

                # Only include males (M) or unknown (U) sex birds
                if sex in ['M', 'U']:
                    target_birds[bird_name] = sex
                    # Generate all variations for this bird name and map them back
                    variations = generate_bird_name_variations(bird_name)
                    for variation in variations:
                        target_bird_variations[variation.lower()] = bird_name
                else:
                    logging.info(f"Skipping {bird_name} - gender: {sex}")
            else:
                # If bird not found in DB, assume unknown and include
                target_birds[bird_name] = 'U'
                # Generate all variations for this bird name and map them back
                variations = generate_bird_name_variations(bird_name)
                for variation in variations:
                    target_bird_variations[variation.lower()] = bird_name

        except Exception as e:
            logging.error(f"Error checking gender for {bird_name}: {e}")
            # If error, assume unknown and include
            target_birds[bird_name] = 'U'
            # Generate all variations for this bird name and map them back
            variations = generate_bird_name_variations(bird_name)
            for variation in variations:
                target_bird_variations[variation.lower()] = bird_name

    con.close()

    logging.info(f"Filtering to {len(target_birds)} male/unknown birds from {len(cross_fostered_birds)} total")
    logging.info(f"Generated {len(target_bird_variations)} name variations for matching")

    try:
        with open(txt_file_path, 'r') as file:
            for line in file:
                line = line.strip()
                if line:  # Skip empty lines
                    # Split on comma to get bird name (first element)
                    parts = line.split(',')
                    if len(parts) >= 4:  # Ensure we have all expected parts
                        txt_file_bird_name = parts[0].strip()

                        # Generate all variations of the bird name from the text file
                        txt_variations = generate_bird_name_variations(txt_file_bird_name)

                        # Check if any variation of the txt file bird name matches any target bird
                        matched_target_bird = None
                        for txt_variation in txt_variations:
                            if txt_variation.lower() in target_bird_variations:
                                matched_target_bird = target_bird_variations[txt_variation.lower()]
                                break

                        if matched_target_bird:
                            # Extract the file path string (last element, remove brackets)
                            file_path_string = parts[3].strip('[]')

                            # Split on '//' to get individual file paths
                            # Filter out empty strings that might result from splitting
                            file_paths = [path for path in file_path_string.split('//') if path.strip()]

                            # Add to dictionary using the original target bird name as key
                            # If we already have entries for this bird, extend the list
                            if matched_target_bird in bird_files:
                                bird_files[matched_target_bird].extend(file_paths)
                            else:
                                bird_files[matched_target_bird] = file_paths

                            logging.info(
                                f"  Matched '{txt_file_bird_name}' from file to target bird '{matched_target_bird}'")
    except FileNotFoundError:
        logging.error(f"File {txt_file_path} not found!")
        return {}

    # Remove duplicates from file paths for each bird
    for bird_name in bird_files:
        bird_files[bird_name] = list(set(bird_files[bird_name]))

    logging.info(f"Found file locations for {len(bird_files)} birds")
    return bird_files

In [3]:
bird_file_locations = get_file_locations_for_birds(birds, txt_file_path='../refs/MacawAllDirsByBird.txt', db_path='../refs/2026-01-15-db.sqlite3')


In [4]:
bird_file_locations

{'ye25': ['brad\\from_dave\\benchmark_learned_song\\y25\\y25_120110_054126.wav',
  'dmets\\from_stork\\dmets\\y25\\05Feb2013_UDFD\\y25_060213_1151.12098.wav',
  'dmets\\papers\\speed_paper\\speed_figs_4_4\\y25_120110_054126.wav',
  'public\\from_egret\\egret\\miller\\LPPI_tutoring\\y25\\02-12-2009\\y25_021209_111320.wav',
  'dmets\\from_egret\\dmets\\striata_recovered_files\\Desktop\\fosters_from_screening_songs\\y25\\y25_120110_054248.wav',
  'public\\from_egret\\egret\\miller\\LPPI_tutoring\\y25\\04-12-2009\\y25_041209_062222.wav',
  'public\\from_egret\\egret\\miller\\LPPI_tutoring\\y25\\26-11-2009\\y25_261109_054001.wav',
  'public\\from_stork\\stork\\tutors-of-colony-nests\\N106-tutor-y25\\02-12-2009\\y25_021209_111719.wav',
  'public\\from_stork\\stork\\tutors-of-colony-nests\\N106-tutor-y25\\03-12-2009\\y25_031209_111951.wav',
  'public\\from_stork\\stork\\tutors-of-colony-nests\\N106-tutor-y25\\18-01-2010\\y25_180110_112109.wav',
  'public\\from_egret\\egret\\miller\\LPPI_tutor

In [5]:
for bird, files in bird_file_locations.items():
    print(f'{bird}: {len(files)}')

ye25: 29
bk61bk26: 34
or14or56: 90
rd92gr54: 16
pu1wh14: 12
pk15bk43: 15
bk13bk12: 5
bk1bk3: 5
bk30bk40: 8
bk31bk89: 30
bk34bk51: 10
bk36bk37: 2
bk38bk39: 2
bk41bk14: 34
bk43bk70: 3
bk4bk47: 2
bk5bk60: 18
bk67bk44: 25
bk73bk71: 2
bk83bk48: 11
bk83bk73: 12
bk89bk88: 4
bk92bk23: 71
bk94bk87: 16
bu28or67: 101
gr26gr95: 4
gr34gr22: 120
gr37gr41: 12
gr82gr37: 34
gr89gr73: 12
gr99or87: 5
or67or64: 69
or76bk11: 5
or8bk49: 5
pk18gr79: 5
pk24bu3: 2
pk36gr15: 6
pk54gr20: 15
pk74gr77: 5
pk97rd22: 5
pu10bk94: 117
pu11bk28: 103
pu27wh16: 5
pu39wh79: 3
pu51wh43: 5
pu61bk41: 77
pu71wh42: 3
pu81bk1: 15
rd10wh46: 89
rd26wh34: 73
rd28wh69: 23
rd2wh49: 2
rd61wh68: 43
rd62wh35: 22
rd63wh86: 12
rd95wh29: 5
wh21pk15: 15
wh26pk33: 54
wh57pk61: 93
wh58pk1: 17
wh67pk20: 30
wh69pk82: 4
wh82pk41: 5
wh94or11: 21
ye1: 17
ye27ye33: 170
bk100bk99: 5
bk1bk9: 4
bk20bk45: 9
bk28bk68: 16
bk29bk8: 189
bk34bk33: 4
bk47bk40: 36
bk4bk95: 34
bk50bk49: 4
bk51bk59: 65
bk5bk25: 7
bk64bk65: 40
bk79bk82: 52
bk86bk62: 2
bu11or68: 

In [8]:
from tools.system_utils import check_sys_for_macaw_root
from x_foster_metadata import get_all_audio_files_for_birds, load_bird_audio_data
root_dir = check_sys_for_macaw_root()
save_file = "cross_fostered_bird_audio_data.json"

In [6]:
all_files = get_all_audio_files_for_birds(bird_file_locations, root_directory=root_dir, save_file=save_file)

In [7]:
bird_audio_data = load_bird_audio_data(save_file)

In [26]:
import pandas as pd
bird_audio_data_df = pd.DataFrame.from_dict(bird_audio_data, orient='index')
bird_audio_data_df.to_csv('E:\daves_xfoster_audio_data.csv')

In [27]:
bird_audio_data_df.keys()

Index(['directories_searched', 'audio_files', 'wav_files', 'cbin_files',
       'batch_files', 'audio_from_batch', 'wav_count', 'cbin_count',
       'wav_and_cbin_count', 'batch_file_count', 'batch_audio_count'],
      dtype='object')

In [28]:
# Create a summary with counts instead of full lists
summary_data = {}

for index, row in bird_audio_data_df.iterrows():
    summary_data[index] = {
        'directories_searched': row['directories_searched'],
        'wav_count': row['wav_count'],
        'cbin_count': row['cbin_count'],
        'wav_and_cbin_count': row['wav_and_cbin_count'],
        'batch_file_count': row['batch_file_count'],
        'batch_audio_count': row['batch_audio_count'],
    }

summary_df = pd.DataFrame.from_dict(summary_data, orient='index')
summary_df.to_csv('E:/daves_xfoster_audio_data_summary.csv')

In [39]:
save_file = "cross_fostered_bird_audio_data_deduplicated_enhanced.json"
dedup_data = load_bird_audio_data(save_file)

In [42]:
dedup_data_df = pd.DataFrame.from_dict(dedup_data, orient='index')
dedup_data_df.to_csv('E:/daves_xfoster_audio_data_deduplicated_enhanced.csv')

In [43]:
dedup_data_df.keys()

Index(['directories_searched', 'audio_files', 'wav_files', 'cbin_files',
       'batch_files', 'audio_from_batch', 'wav_count', 'cbin_count',
       'wav_and_cbin_count', 'batch_file_count', 'batch_audio_count',
       'audio_files_info', 'wav_files_info', 'cbin_files_info',
       'batch_files_info', 'audio_from_batch_info'],
      dtype='object')

In [46]:
# Create a summary with counts instead of full lists
summary_data = {}

for index, row in dedup_data_df.iterrows():
    summary_data[index] = {
        'directories_searched': row['directories_searched'],
        'wav_count': row['wav_count'],
        'cbin_count': row['cbin_count'],
        'wav_and_cbin_count': row['wav_and_cbin_count'],
        'batch_file_count': row['batch_file_count'],
        'batch_audio_count': row['batch_audio_count'],
    }

summary_df = pd.DataFrame.from_dict(summary_data, orient='index')
summary_df.to_csv('E:/daves_xfoster_audio_data_deduplicated_enhanced_summary.csv')

In [45]:
dedup_data_df['audio_files_info']

ye25        [{'filepath': '\\macaw.ucsf.edu\users\brad\fro...
bk61bk26    [{'filepath': '\\macaw.ucsf.edu\users\brad\tut...
or14or56    [{'filepath': '\\macaw.ucsf.edu\users\brad\tut...
rd92gr54    [{'filepath': '\\macaw.ucsf.edu\users\brad\tut...
pu1wh14     [{'filepath': '\\macaw.ucsf.edu\users\public\f...
                                  ...                        
wh39or13    [{'filepath': '\\macaw.ucsf.edu\users\public\f...
bk9bk8      [{'filepath': '\\macaw.ucsf.edu\users\public\f...
wh80or5     [{'filepath': '\\macaw.ucsf.edu\users\public\f...
rd93wh89    [{'filepath': '\\macaw.ucsf.edu\users\public\f...
rd3wh90     [{'filepath': '\\macaw.ucsf.edu\users\public\f...
Name: audio_files_info, Length: 167, dtype: object